In [14]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Unlox/Assignment2/rahul_transactions.csv')
print("Raw shape:", df.shape)
df.head()
df['date'] = pd.to_datetime(df['Date'], errors='coerce', dayfirst=True)

df['amount'] = (
    df['Amount']
    .astype(str)
    .str.replace('₹', '', regex=False)
    .str.replace('Rs.', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')


type_map = {'DR': 'debit', 'CR': 'credit', 'Debit': 'debit', 'Credit': 'credit'}
df['type_clean'] = df['Type'].map(type_map)

before = len(df)
df = df.drop_duplicates()
dropped = before - len(df)

bad_dates = df['date'].isna().sum()
bad_amounts = df['amount'].isna().sum()

print(f"Parsed {len(df)} transactions across 6 months.\n")
print(f"Dropped {dropped} duplicates.\n")
print(f"{bad_dates} unparseable dates, {bad_amounts} unparseable amounts.\n")
VENDOR_KEYWORDS = {
    'Amazon Prime':      ['PRIME VIDEO', 'AMZN PRIME', 'AMAZON-PRIME', 'AMAZON PRIME'],
    'Swiggy Instamart':  ['INSTAMART'],
    'Swiggy':            ['SWIGGY', 'BUNDL'],
    'Zomato':            ['ZOMATO'],
    'Blinkit':           ['BLINKIT', 'GROFERS'],
    'Zepto':             ['ZEPTO', 'KIRANAKART'],
    'BigBasket':         ['BIGBASKET', 'INNOVATIVE RETAIL'],
    'DMart':             ['DMART', 'AVENUE SUPERMARTS'],
    'Flipkart':          ['FLIPKART', 'FKART'],
    'Myntra':            ['MYNTRA'],
    'Nykaa':             ['NYKAA', 'FSN E-COMMERCE'],
    'Amazon':            ['AMAZON', 'AMZN'],
    'Uber':              ['UBER'],
    'Ola':               ['OLA', 'ANI TECHNOLOGIES', 'ROPPEN'],
    'Rapido':            ['RAPIDO'],
    'BMTC':              ['BMTC', 'TUMMOC'],
    'Zerodha':           ['ZERODHA'],
    'Groww':             ['GROWW', 'NEXTBILLION'],
    'Netflix':           ['NETFLIX'],
    'Spotify':           ['SPOTIFY'],
    'Hotstar':           ['HOTSTAR', 'STAR INDIA'],
    'Airtel':            ['AIRTEL', 'BHARTI'],
    'Vi':                ['UPI-VI-', 'VI POSTPAID', 'VODAFONE'],
    'Jio':               ['JIO'],
    'BESCOM':            ['BESCOM', 'BANGALORE ELEC'],
    'BWSSB':             ['BWSSB'],
    'BPCL':              ['BPCL'],
    'HP Petrol':         ['HP PETROL'],
    'Indian Oil':        ['INDIAN OIL', 'IOC'],
    'Cafe Coffee Day':   ['COFFEE DAY', 'UPI-CCD'],
    'Starbucks':         ['STARBUCKS'],
    'Third Wave Coffee': ['THIRDWAVE', 'THIRD WAVE', 'TWC INDIA'],
    'BookMyShow':        ['BOOKMYSHOW', 'BMS MOVIE', 'BIGTREE'],
    'Meghana Foods':     ['MEGHANA'],
    'Truffles':          ['TRUFFLES'],
    'Empire Restaurant': ['EMPIRE RESTAURANT'],
    'Dineout':           ['DINEOUT'],
    'Restaurant':        ['RESTAURANT'],
    'Rent':              ['RENT-LANDLORD'],
    'Salary':            ['SALARY'],
    'P2P Transfer':      ['UPI-AMAN', 'UPI-ANKIT', 'UPI-PRIYA', 'UPI-NEHA',
                           'UPI-VIKAS', 'UPI-KARAN', 'UPI-SNEHA'],
    'Cash Withdrawal':   ['ATM-WDL', 'ATM WDL'],
}

def extract_vendor(description):
    desc_upper = str(description).upper()
    for vendor, keywords in VENDOR_KEYWORDS.items():
        for kw in keywords:
            if kw in desc_upper:
                return vendor
    return 'Uncategorised'

df['vendor_clean'] = df['Description'].apply(extract_vendor)

print("Unique canonical vendors:", df['vendor_clean'].nunique())
print(df['vendor_clean'].value_counts().head(10))

uncategorised_descriptions = sorted(df[df['vendor_clean'] == 'Uncategorised']['Description'].unique())
print(f"Uncategorised: {len(uncategorised_descriptions)}\n")
CATEGORY_MAP = {
    'Swiggy': 'Food Delivery', 'Zomato': 'Food Delivery',
    'Meghana Foods': 'Restaurants', 'Truffles': 'Restaurants',
    'Empire Restaurant': 'Restaurants', 'Dineout': 'Restaurants', 'Restaurant': 'Restaurants',
    'Swiggy Instamart': 'Quick Commerce', 'Blinkit': 'Quick Commerce', 'Zepto': 'Quick Commerce',
    'BigBasket': 'Groceries', 'DMart': 'Groceries',
    'Amazon': 'Ecommerce', 'Flipkart': 'Ecommerce', 'Myntra': 'Ecommerce', 'Nykaa': 'Ecommerce',
    'Uber': 'Transport', 'Ola': 'Transport', 'Rapido': 'Transport', 'BMTC': 'Transport',
    'Zerodha': 'Investments', 'Groww': 'Investments',
    'Netflix': 'Subscriptions', 'Spotify': 'Subscriptions', 'Hotstar': 'Subscriptions',
    'Amazon Prime': 'Subscriptions',
    'Airtel': 'Utilities', 'Vi': 'Utilities', 'Jio': 'Utilities',
    'BESCOM': 'Utilities', 'BWSSB': 'Utilities',
    'BPCL': 'Fuel', 'HP Petrol': 'Fuel', 'Indian Oil': 'Fuel',
    'Cafe Coffee Day': 'Cafe', 'Starbucks': 'Cafe', 'Third Wave Coffee': 'Cafe',
    'BookMyShow': 'Entertainment',
    'P2P Transfer': 'Personal Transfer', 'Cash Withdrawal': 'Cash Withdrawal',
    'Rent': 'Rent', 'Salary': 'Income',
}
df['category'] = df['vendor_clean'].map(CATEGORY_MAP).fillna('Uncategorised')

print(df['category'].value_counts())


Raw shape: (1328, 8)
Parsed 1310 transactions across 6 months.

Dropped 18 duplicates.

1167 unparseable dates, 0 unparseable amounts.

Unique canonical vendors: 42
vendor_clean
Swiggy              176
Zomato              121
Ola                 101
Amazon               76
Uber                 71
Zepto                71
Swiggy Instamart     67
Blinkit              55
Flipkart             47
Starbucks            42
Name: count, dtype: int64
Uncategorised: 0

category
Food Delivery        297
Transport            250
Quick Commerce       193
Ecommerce            162
Cafe                  99
Restaurants           73
Utilities             43
Groceries             41
Subscriptions         41
Fuel                  28
Investments           23
Personal Transfer     18
Cash Withdrawal       17
Entertainment         13
Income                 6
Rent                   6
Name: count, dtype: int64


In [15]:
NON_DISCRETIONARY = {'Personal Transfer', 'Cash Withdrawal', 'Rent', 'Income', 'Uncategorised'}

total_credits = df.loc[df['type_clean'] == 'credit', 'amount'].sum()
total_debits = df.loc[df['type_clean'] == 'debit', 'amount'].sum()
net_change = total_credits - total_debits
savings_rate = (total_credits - total_debits) / total_credits * 100

spend_df = df[(df['type_clean'] == 'debit') & (~df['category'].isin(NON_DISCRETIONARY))].copy()
category_totals = spend_df.groupby('category')['amount'].sum().sort_values(ascending=False)
category_pct = category_totals / category_totals.sum() * 100
vendor_totals = spend_df.groupby('vendor_clean')['amount'].agg(total='sum', orders='count')
vendor_totals = vendor_totals.sort_values('total', ascending=False)

print(f"Total credits: Rs. {total_credits:,.0f}")
print(f"Total debits: Rs. {total_debits:,.0f}")
print(f"Net change: Rs. {net_change:,.0f}")
print(f"Savings rate: {savings_rate:.1f}%")
print(category_pct.head(5))
print(vendor_totals.head(5))

spend_df['month'] = spend_df['date'].dt.month
month_pivot = spend_df.pivot_table(values='amount', index='category', columns='month',
                                    aggfunc='sum', fill_value=0)

first_m, last_m = month_pivot.columns.min(), month_pivot.columns.max()
growth_pct = (month_pivot[last_m] - month_pivot[first_m]) / month_pivot[first_m].replace(0, np.nan) * 100
biggest_growth_cat = growth_pct.idxmax()
biggest_decline_cat = growth_pct.idxmin()

print(f"Biggest growth: {biggest_growth_cat} ({growth_pct.max():.1f}%)")
print(f"Biggest decline: {biggest_decline_cat} ({growth_pct.min():.1f}%)\n")
df['hour'] = df['Time'].str[:2].astype(int)
spend_df['hour'] = df.loc[spend_df.index, 'hour']

food = df[df['category'] == 'Food Delivery']
late_night_food = food[(food['hour'] >= 21) | (food['hour'] <= 1)]
late_night_pct = len(late_night_food) / len(food) * 100

cafe = df[df['category'] == 'Cafe']
morning_cafe = cafe[(cafe['hour'] >= 8) & (cafe['hour'] <= 11)]
morning_cafe_pct = len(morning_cafe) / len(cafe) * 100

# category x hour NumPy matrix
categories_list = sorted(spend_df['category'].unique())
hour_matrix = np.zeros((len(categories_list), 24))
for i, cat in enumerate(categories_list):
    cat_hours = df.loc[df['category'] == cat, 'hour']
    for h in range(24):
        hour_matrix[i, h] = (cat_hours == h).sum()

print(f"Food Delivery late-night share: {late_night_pct:.1f}%")
print(f"Cafe morning share: {morning_cafe_pct:.1f}%")


Total credits: Rs. 509,774
Total debits: Rs. 1,678,901
Net change: Rs. -1,169,127
Savings rate: -229.3%
category
Ecommerce         39.563447
Investments       16.535159
Food Delivery      8.599002
Restaurants        7.844939
Quick Commerce     6.374392
Name: amount, dtype: float64
                 total  orders
vendor_clean                  
Amazon        318422.0      76
Zerodha       210000.0      14
Flipkart      177510.0      47
Swiggy         73738.0     176
Myntra         69529.0      20
Biggest growth: Food Delivery (355.1%)
Biggest decline: Quick Commerce (-100.0%)

Food Delivery late-night share: 20.5%
Cafe morning share: 35.4%


In [16]:
cat_mean = spend_df.groupby('category')['amount'].transform('mean')
cat_std = spend_df.groupby('category')['amount'].transform('std')
spend_df['z_score'] = (spend_df['amount'] - cat_mean) / cat_std
anomalies = spend_df[spend_df['z_score'] > 2].sort_values('z_score', ascending=False)

print(f"Anomalies flagged: {len(anomalies)}\n")
print(anomalies[['date', 'vendor_clean', 'category', 'amount', 'z_score']].head(5))
def pct_of_debits(categories):
    return category_totals.reindex(categories).fillna(0).sum() / category_totals.sum() * 100

def detect_archetypes():
    found = []
    foodie_pct = pct_of_debits(['Food Delivery', 'Restaurants', 'Cafe'])
    if foodie_pct > 25:
        found.append(('THE FOODIE', f'{foodie_pct:.1f}% on food'))
    qc_pct = pct_of_debits(['Quick Commerce'])
    if qc_pct > 15:
        found.append(('THE QUICK COMMERCE JUNKIE', f'{qc_pct:.1f}% on Q-com'))
    ecom_pct = pct_of_debits(['Ecommerce'])
    if ecom_pct > 15:
        found.append(('THE SHOPAHOLIC', f'{ecom_pct:.1f}% on e-commerce'))
    inv_pct = pct_of_debits(['Investments'])
    if inv_pct > 15:
        found.append(('THE INVESTOR', f'{inv_pct:.1f}% on SIPs'))
    if late_night_pct > 50:
        found.append(('THE LATE-NIGHT SNACKER', f'{late_night_pct:.1f}% food after 9PM'))
    transport_pct = pct_of_debits(['Transport'])
    if transport_pct > 10:
        found.append(('THE CAB COMMUTER', f'{transport_pct:.1f}% on transport'))
    sub_vendors = df.loc[df['category'] == 'Subscriptions', 'vendor_clean'].nunique()
    if sub_vendors >= 5:
        found.append(('THE SUBSCRIPTION LOVER', f'{sub_vendors} active subs'))
    if savings_rate < 10:
        found.append(('THE YOLO SPENDER', f'savings rate {savings_rate:.1f}%'))
    if savings_rate > 40:
        found.append(('THE DISCIPLINED SAVER', f'savings rate {savings_rate:.1f}%'))
    return found

archetypes = detect_archetypes()
for name, metric in archetypes:
    print(f" -> {name} ({metric})")


Anomalies flagged: 24

     date   vendor_clean     category   amount   z_score
1298  NaT         Amazon    Ecommerce  22008.0  3.974482
269   NaT         Amazon    Ecommerce  21986.0  3.969715
414   NaT     Restaurant  Restaurants   8383.0  3.884639
1271  NaT        Dineout  Restaurants   7935.0  3.627582
653   NaT  Meghana Foods  Restaurants   7931.0  3.625287
 -> THE SHOPAHOLIC (39.6% on e-commerce)
 -> THE INVESTOR (16.5% on SIPs)
 -> THE YOLO SPENDER (savings rate -229.3%)
